In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("../data/analysis_data.csv")

In [3]:
data.shape

(2530, 30)

In [4]:
data.columns.tolist()

['SEQN',
 'TELOMEAN',
 'TELOSTD',
 'HorvathTelo',
 'RIDAGEYR',
 'qPCR_age_pred',
 'qPCR_residual',
 'DNAm_age_pred',
 'DNAm_residual',
 'qPCR_z',
 'DNAm_z',
 'discordance_signed',
 'discordance',
 'TELO_CV',
 'WBC',
 'lymphocyte_pct',
 'monocyte_pct',
 'neutrophil_pct',
 'eosinophil_pct',
 'basophil_pct',
 'CRP',
 'BMXBMI',
 'BMXWAIST',
 'SMQ020',
 'SMQ040',
 'SMD070',
 'smoking_status',
 'RIAGENDR',
 'sex',
 'HbA1c']

In [5]:
feature_columns = [
    "WBC",
    "lymphocyte_pct",
    "monocyte_pct",
    "neutrophil_pct",
    "eosinophil_pct",
    "basophil_pct",
    "CRP",
    "BMXBMI",
    "BMXWAIST",
    "HbA1c",
    "smoking_status",
    "sex"
]

In [6]:
target_source_columns = [
    "RIDAGEYR",
    "TELOMEAN",
    "HorvathTelo"
]

In [7]:
modeling_data = data[
    ["SEQN"] + target_source_columns + feature_columns
].copy()

In [8]:
modeling_data.shape

(2530, 16)

In [9]:
modeling_data.isna().sum()

SEQN                0
RIDAGEYR            0
TELOMEAN            0
HorvathTelo         0
WBC                 3
lymphocyte_pct     13
monocyte_pct       13
neutrophil_pct     13
eosinophil_pct     13
basophil_pct       13
CRP                 0
BMXBMI            117
BMXWAIST          117
HbA1c               1
smoking_status      6
sex                 0
dtype: int64

In [10]:
from sklearn.model_selection import train_test_split

In [11]:
development_data, test_data = train_test_split(
    modeling_data, 
    test_size=0.20,
    random_state=42
)

In [12]:
train_data, val_data = train_test_split(
    development_data,
    test_size=0.20,
    random_state=42
)

In [13]:
print("Train:", train_data.shape)
print("Validation:", val_data.shape)
print("Test:", test_data.shape)

Train: (1619, 16)
Validation: (405, 16)
Test: (506, 16)


In [14]:
from sklearn.linear_model import LinearRegression

In [15]:
qpcr_age_model = LinearRegression()

In [16]:
qpcr_age_model.fit(
    train_data[["RIDAGEYR"]],
    train_data["TELOMEAN"]
)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](1,)",[-0.01]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](1,)",['RIDAGEYR']
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,1.277
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,1
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,1


In [17]:
print("Slope:", qpcr_age_model.coef_)
print("Intrcept:", qpcr_age_model.intercept_)

Slope: [-0.00536888]
Intrcept: 1.276719062086414


In [18]:
train_data["qPCR_age_pred"] = qpcr_age_model.predict(
    train_data[["RIDAGEYR"]])

In [19]:
val_data["qPCR_age_pred"] = qpcr_age_model.predict(
    val_data[["RIDAGEYR"]]
)

In [20]:
test_data["qPCR_age_pred"] = qpcr_age_model.predict(
    test_data[["RIDAGEYR"]]
)

In [21]:
train_data["qPCR_residual"] = (
    train_data["TELOMEAN"]
    - train_data["qPCR_age_pred"]
)

val_data["qPCR_residual"] = (
    val_data["TELOMEAN"]
    - val_data["qPCR_age_pred"]
)

test_data["qPCR_residual"] = (
    test_data["TELOMEAN"]
    - test_data["qPCR_age_pred"]
)

In [22]:
train_data[
    ["RIDAGEYR", "TELOMEAN", "qPCR_age_pred", "qPCR_residual"]
].head()

,RIDAGEYR,TELOMEAN,qPCR_age_pred,qPCR_residual
2317,51.0,1.147246,1.002906,0.144340
488,65.0,0.810279,0.927742,-0.117463
1951,71.0,0.949937,0.895529,0.054408
501,85.0,0.817838,0.820364,-0.002526
1903,65.0,0.932364,0.927742,0.004622


In [23]:
train_data[
    ["RIDAGEYR", "qPCR_residual"]
].corr()

,RIDAGEYR,qPCR_residual
RIDAGEYR,1.000000e+00,-7.247408e-16
qPCR_residual,-7.247408e-16,1.000000e+00


In [24]:
from sklearn.linear_model import LinearRegression

dnam_age_model = LinearRegression()

dnam_age_model.fit(
    train_data[["RIDAGEYR"]],
    train_data["HorvathTelo"]
)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](1,)",[-0.02]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](1,)",['RIDAGEYR']
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,7.765
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,1
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,1


In [25]:
train_data["DNAm_age_pred"] = dnam_age_model.predict(
    train_data[["RIDAGEYR"]]
)

val_data["DNAm_age_pred"] = dnam_age_model.predict(
    val_data[["RIDAGEYR"]]
)

test_data["DNAm_age_pred"] = dnam_age_model.predict(
    test_data[["RIDAGEYR"]]
)

In [26]:
train_data["DNAm_residual"] = (
    train_data["HorvathTelo"]
    - train_data["DNAm_age_pred"]
)

val_data["DNAm_residual"] = (
    val_data["HorvathTelo"]
    - val_data["DNAm_age_pred"]
)

test_data["DNAm_residual"] = (
    test_data["HorvathTelo"]
    - test_data["DNAm_age_pred"]
)

In [27]:
qpcr_residual_mean = train_data["qPCR_residual"].mean()
qpcr_residual_std = train_data["qPCR_residual"].std()

In [28]:
dnam_residual_mean = train_data["DNAm_residual"].mean()
dnam_residual_std = train_data["DNAm_residual"].std()

In [29]:
print("qPCR mean:", qpcr_residual_mean)
print("qPCR SD:", qpcr_residual_std)

print("DNAm mean:", dnam_residual_mean)
print("DNAm SD:", dnam_residual_std)

qPCR mean: 1.1904553247370426e-16
qPCR SD: 0.2182593105717395
DNAm mean: -5.760267700340528e-17
DNAm SD: 0.2559525114947862


In [30]:
train_data["qPCR_z"] = (
    train_data["qPCR_residual"] - qpcr_residual_mean
) / qpcr_residual_std

val_data["qPCR_z"] = (
    val_data["qPCR_residual"] - qpcr_residual_mean
) / qpcr_residual_std

test_data["qPCR_z"] = (
    test_data["qPCR_residual"] - qpcr_residual_mean
) / qpcr_residual_std

In [31]:
train_data["DNAm_z"] = (
    train_data["DNAm_residual"] - dnam_residual_mean
) / dnam_residual_std

val_data["DNAm_z"] = (
    val_data["DNAm_residual"] - dnam_residual_mean
) / dnam_residual_std

test_data["DNAm_z"] = (
    test_data["DNAm_residual"] - dnam_residual_mean
) / dnam_residual_std

In [32]:
train_data["discordance_signed"] = (
    train_data["qPCR_z"] - train_data["DNAm_z"]
)

val_data["discordance_signed"] = (
    val_data["qPCR_z"] - val_data["DNAm_z"]
)

test_data["discordance_signed"] = (
    test_data["qPCR_z"] - test_data["DNAm_z"]
)

In [33]:
train_data["discordance"] = train_data["discordance_signed"].abs()
val_data["discordance"] = val_data["discordance_signed"].abs()
test_data["discordance"] = test_data["discordance_signed"].abs()

In [34]:
train_data[["qPCR_z", "DNAm_z"]].describe()

,qPCR_z,DNAm_z
count,1.619000e+03,1.619000e+03
mean,1.316633e-17,2.742985e-18
std,1.000000e+00,1.000000e+00
min,-2.396351e+00,-3.938964e+00
25%,-6.713681e-01,-5.953923e-01
50%,-1.119227e-01,1.591451e-02
75%,5.343133e-01,6.240840e-01
max,5.609237e+00,5.272312e+00


In [35]:
train_data[
    ["discordance_signed", "discordance"]
].describe()

,discordance_signed,discordance
count,1.619000e+03,1619.000000
mean,1.097194e-18,0.911859
std,1.191735e+00,0.766963
min,-5.243194e+00,0.001966
25%,-7.569241e-01,0.348904
50%,-5.757758e-02,0.731398
75%,6.904166e-01,1.269823
max,5.711467e+00,5.711467


In [36]:
X_train = train_data[feature_columns].copy()
X_val = val_data[feature_columns].copy()
X_test = test_data[feature_columns].copy()

y_train = train_data["discordance_signed"].copy()
y_val = val_data["discordance_signed"].copy()
y_test = test_data["discordance_signed"].copy()

In [37]:
print("X train:", X_train.shape)
print("y train:", y_train.shape)

print("X val:", X_val.shape)
print("y val:", y_val.shape)

print("X test:", X_test.shape)
print("y test:", y_test.shape)

X train: (1619, 12)
y train: (1619,)
X val: (405, 12)
y val: (405,)
X test: (506, 12)
y test: (506,)


In [38]:
X_train.isna().sum().sort_values(ascending=False)

BMXBMI            80
BMXWAIST          79
lymphocyte_pct     9
monocyte_pct       9
neutrophil_pct     9
eosinophil_pct     9
basophil_pct       9
WBC                3
smoking_status     3
HbA1c              1
CRP                0
sex                0
dtype: int64

In [39]:
print("Missing y_train:", y_train.isna().sum())
print("Missing y_val:", y_val.isna().sum())
print("Missing y_test:", y_test.isna().sum())

Missing y_train: 0
Missing y_val: 0
Missing y_test: 0


In [40]:
numeric_features = [
    "WBC",
    "lymphocyte_pct",
    "monocyte_pct",
    "neutrophil_pct",
    "eosinophil_pct",
    "basophil_pct",
    "CRP",
    "BMXBMI",
    "BMXWAIST",
    "HbA1c"
]

categorical_features = [
    "smoking_status",
    "sex"
]

In [41]:
X_train["body_measure_missing"] = (
    X_train["BMXBMI"].isna() &
    X_train["BMXWAIST"].isna()
).astype(int)

X_val["body_measure_missing"] = (
    X_val["BMXBMI"].isna() &
    X_val["BMXWAIST"].isna()
).astype(int)

X_test["body_measure_missing"] = (
    X_test["BMXBMI"].isna() &
    X_test["BMXWAIST"].isna()
).astype(int)

In [42]:
from sklearn.impute import SimpleImputer

In [43]:
numeric_imputer = SimpleImputer(strategy="median")

In [44]:
numeric_imputer.fit(
    X_train[numeric_features]
)

,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](10,)","['WBC','lymphocyte_pct','monocyte_pct',...,'BMXBMI','BMXWAIST','HbA1c']"
indicator_ indicator_: :class:`~sklearn.impute.MissingIndicator`Indicator used to add binary indicators for missing values.`None` if `add_indicator=False`.,NoneType,None
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,10
"statistics_ statistics_: array of shape (n_features,)The imputation fill value for each feature.Computing statistics can result in `np.nan` values.During :meth:`transform`, features corresponding to `np.nan`statistics will be discarded.","ndarray[float64](10,)","[ 6.8 ,29.65, 8.1 ,...,27.94,99.25, 5.6 ]"


In [45]:
X_train[numeric_features] = numeric_imputer.transform(
    X_train[numeric_features]
)

In [46]:
X_val[numeric_features] = numeric_imputer.transform(
    X_val[numeric_features]
)

In [47]:
X_test[numeric_features] = numeric_imputer.transform(
    X_test[numeric_features]
)

In [48]:
X_train[numeric_features].isna().sum()

WBC               0
lymphocyte_pct    0
monocyte_pct      0
neutrophil_pct    0
eosinophil_pct    0
basophil_pct      0
CRP               0
BMXBMI            0
BMXWAIST          0
HbA1c             0
dtype: int64

In [49]:
X_val[numeric_features].isna().sum()
X_test[numeric_features].isna().sum()

WBC               0
lymphocyte_pct    0
monocyte_pct      0
neutrophil_pct    0
eosinophil_pct    0
basophil_pct      0
CRP               0
BMXBMI            0
BMXWAIST          0
HbA1c             0
dtype: int64

In [50]:
from sklearn.impute import SimpleImputer

categorical_imputer = SimpleImputer(
    strategy="constant",
    fill_value="missing"
)

In [51]:
categorical_imputer.fit(
    X_train[categorical_features]
)

,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'constant'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",'missing'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](2,)","['smoking_status','sex']"
indicator_ indicator_: :class:`~sklearn.impute.MissingIndicator`Indicator used to add binary indicators for missing values.`None` if `add_indicator=False`.,NoneType,None
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,2
"statistics_ statistics_: array of shape (n_features,)The imputation fill value for each feature.Computing statistics can result in `np.nan` values.During :meth:`transform`, features corresponding to `np.nan`statistics will be discarded.","ndarray[object](2,)","['missing','missing']"


In [52]:
train_cat = categorical_imputer.transform(
    X_train[categorical_features]
)

val_cat = categorical_imputer.transform(
    X_val[categorical_features]
)

test_cat = categorical_imputer.transform(
    X_test[categorical_features]
)

In [53]:
from sklearn.preprocessing import OneHotEncoder

In [54]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

In [55]:
encoder.fit(train_cat)

,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide <encoder_infreque

In [56]:
train_cat_encoded = encoder.transform(train_cat)
val_cat_encoded = encoder.transform(val_cat)
test_cat_encoded = encoder.transform(test_cat)

In [57]:
encoded_columns = encoder.get_feature_names_out(
    categorical_features
)

encoded_columns

array(['smoking_status_current', 'smoking_status_former',
       'smoking_status_missing', 'smoking_status_never', 'sex_female',
       'sex_male'], dtype=object)

In [58]:
train_cat_encoded = pd.DataFrame(
    train_cat_encoded,
    columns=encoded_columns,
    index=X_train.index
)

val_cat_encoded = pd.DataFrame(
    val_cat_encoded,
    columns=encoded_columns,
    index=X_val.index
)

test_cat_encoded = pd.DataFrame(
    test_cat_encoded,
    columns=encoded_columns,
    index=X_test.index
)

In [59]:
numeric_with_indicator = (
    numeric_features + ["body_measure_missing"]
)

In [60]:
X_train_processed = pd.concat(
    [
        X_train[numeric_with_indicator],
        train_cat_encoded
    ],
    axis=1
)

In [61]:
X_val_processed = pd.concat(
    [
        X_val[numeric_with_indicator],
        val_cat_encoded
    ],
    axis=1
)

In [62]:
X_test_processed = pd.concat(
    [
        X_test[numeric_with_indicator],
        test_cat_encoded
    ],
    axis=1
)

In [63]:
X_train_processed.head()

,WBC,lymphocyte_pct,monocyte_pct,neutrophil_pct,eosinophil_pct,basophil_pct,CRP,BMXBMI,BMXWAIST,HbA1c,body_measure_missing,smoking_status_current,smoking_status_former,smoking_status_missing,smoking_status_never,sex_female,sex_male
2317,5.3,27.7,13.1,57.4,1.4,0.4,0.10,27.53,93.7,5.3,0,0.0,0.0,0.0,1.0,0.0,1.0
488,4.9,28.8,6.9,60.2,3.2,0.9,0.66,30.22,98.9,8.3,0,0.0,0.0,0.0,1.0,1.0,0.0
1951,7.6,31.3,7.6,59.1,0.9,1.2,1.12,20.33,83.5,5.1,0,1.0,0.0,0.0,0.0,1.0,0.0
501,6.9,24.6,8.5,64.0,1.9,1.0,0.15,17.87,69.6,5.1,0,0.0,1.0,0.0,0.0,1.0,0.0
1903,6.2,33.3,8.1,55.1,2.6,1.0,0.25,31.44,107.0,5.3,0,0.0,0.0,0.0,1.0,0.0,1.0


In [64]:
X_train_processed.dtypes

WBC                       float64
lymphocyte_pct            float64
monocyte_pct              float64
neutrophil_pct            float64
eosinophil_pct            float64
basophil_pct              float64
CRP                       float64
BMXBMI                    float64
BMXWAIST                  float64
HbA1c                     float64
body_measure_missing        int64
smoking_status_current    float64
smoking_status_former     float64
smoking_status_missing    float64
smoking_status_never      float64
sex_female                float64
sex_male                  float64
dtype: object

In [65]:
X_train_processed.isna().sum().sum()

np.int64(0)

In [66]:
from sklearn.preprocessing import StandardScaler

In [67]:
scaler = StandardScaler()

In [68]:
scaler.fit(
    X_train_processed[numeric_features]
)

StandardScaler()

In [69]:
X_train_processed[numeric_features] = scaler.transform(
    X_train_processed[numeric_features]
)

In [70]:
X_val_processed[numeric_features] = scaler.transform(
    X_val_processed[numeric_features]
)

In [71]:
X_test_processed[numeric_features] = scaler.transform(
    X_test_processed[numeric_features]
)

In [72]:
X_train_processed[numeric_features].describe()

,WBC,lymphocyte_pct,monocyte_pct,neutrophil_pct,eosinophil_pct,basophil_pct,CRP,BMXBMI,BMXWAIST,HbA1c
count,1.619000e+03,1.619000e+03,1.619000e+03,1.619000e+03,1.619000e+03,1.619000e+03,1.619000e+03,1.619000e+03,1.619000e+03,1.619000e+03
mean,-2.106612e-16,-5.266530e-17,9.326148e-17,6.780658e-16,1.206913e-16,-1.536071e-17,-8.119234e-17,6.912321e-16,-5.326876e-16,-7.548694e-16
std,1.000309e+00,1.000309e+00,1.000309e+00,1.000309e+00,1.000309e+00,1.000309e+00,1.000309e+00,1.000309e+00,1.000309e+00,1.000309e+00
min,-2.377588e+00,-2.788041e+00,-2.885812e+00,-3.786287e+00,-1.312748e+00,-1.522741e+00,-4.395462e-01,-2.185787e+00,-2.739858e+00,-1.437180e+00
25%,-7.051628e-01,-6.988266e-01,-6.559499e-01,-6.361092e-01,-6.052006e-01,-5.961040e-01,-3.496520e-01,-6.633629e-01,-6.477651e-01,-5.058413e-01
50%,-9.700801e-02,-3.818294e-02,-1.312764e-01,2.898571e-02,-2.072055e-01,-1.327855e-01,-2.297930e-01,-1.237141e-01,-3.617276e-02,-2.730067e-01
75%,5.618263e-01,6.168142e-01,5.245655e-01,6.786133e-01,3.234547e-01,3.305329e-01,2.433826e-03,4.817456e-01,5.866072e-01,3.743954e-02
max,7.809004e+00,4.902528e+00,7.214152e+00,3.328681e+00,1.115777e+01,1.816829e+01,2.172688e+01,5.936147e+00,4.084617e+00,9.971718e+00


In [73]:
from sklearn.dummy import DummyRegressor

In [74]:
dummy_model = DummyRegressor(strategy="mean")

In [75]:
dummy_model.fit(X_train_processed, y_train)

,"strategy strategy: {""mean"", ""median"", ""quantile"", ""constant""}, default=""mean""Strategy to use to generate predictions.* ""mean"": always predicts the mean of the training set* ""median"": always predicts the median of the training set* ""quantile"": always predicts a specified quantile of the training set, provided with the quantile parameter.* ""constant"": always predicts a constant value that is provided by the user.",'mean'
,"constant constant: int or float or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
,"quantile quantile: float in [0.0, 1.0], default=NoneThe quantile to predict using the ""quantile"" strategy. A quantile of0.5 corresponds to the median, while 0.0 to the minimum and 1.0 to themaximum.",None
Name,Type,Value
"constant_ constant_: ndarray of shape (1, n_outputs)Mean or median or quantile of the training targets or constant valuegiven by the user.","ndarray[float64](1, 1)",[[0.]]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](17,)","['WBC','lymphocyte_pct','monocyte_pct',...,'smoking_status_never', 'sex_female','sex_male']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,17
n_outputs_ n_outputs_: intNumber of outputs.,int,1


In [76]:
y_val_pred_dummy = dummy_model.predict(
    X_val_processed
)

In [77]:
y_val_pred_dummy[:10]

array([1.09719385e-18, 1.09719385e-18, 1.09719385e-18, 1.09719385e-18,
       1.09719385e-18, 1.09719385e-18, 1.09719385e-18, 1.09719385e-18,
       1.09719385e-18, 1.09719385e-18])

In [78]:
from sklearn.metrics import mean_absolute_error, r2_score

In [79]:
dummy_mae = mean_absolute_error(
    y_val,
    y_val_pred_dummy
)

dummy_r2 = r2_score(
    y_val,
    y_val_pred_dummy
)

print("Dummy MAE:", dummy_mae)
print("Dummy R²:", dummy_r2)

Dummy MAE: 0.8831646487900602
Dummy R²: -0.012951941137599343


In [80]:
from sklearn.linear_model import Ridge

In [81]:
ridge_model = Ridge(alpha=1.0)

In [82]:
ridge_model.fit(
    X_train_processed,
    y_train
)

,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' 

In [83]:
y_val_pred_ridge = ridge_model.predict(
    X_val_processed
)

In [84]:
ridge_mae = mean_absolute_error(
    y_val,
    y_val_pred_ridge
)

ridge_r2 = r2_score(
    y_val,
    y_val_pred_ridge
)

print("Ridge MAE:", ridge_mae)
print("Ridge R²:", ridge_r2)

Ridge MAE: 0.8872152583086084
Ridge R²: 0.0026535986829079317


In [85]:
alphas = [0.01, 0.1, 1, 10, 100]

In [86]:
for alpha in alphas:
    model = Ridge(alpha=alpha)

    model.fit(
        X_train_processed,
        y_train
    )

    pred = model.predict(
        X_val_processed
    )

    mae = mean_absolute_error(y_val, pred)
    r2 = r2_score(y_val, pred)

    print(
        "alpha =", alpha,
        "| MAE =", mae,
        "| R² =", r2
    )

alpha = 0.01 | MAE = 0.8868801235120535 | R² = 0.0014791178308454223
alpha = 0.1 | MAE = 0.8871394910953045 | R² = 0.0012084657103269558
alpha = 1 | MAE = 0.8872152583086084 | R² = 0.0026535986829079317
alpha = 10 | MAE = 0.8860648552665743 | R² = 0.006784911608955335
alpha = 100 | MAE = 0.8851020175501028 | R² = 0.005953582643950517


In [87]:
best_alpha = 100

In [88]:
import torch

In [89]:
X_train_tensor = torch.tensor(
    X_train_processed.to_numpy(dtype="float32")
)

In [90]:
X_val_tensor = torch.tensor(
    X_val_processed.to_numpy(dtype="float32")
)

X_test_tensor = torch.tensor(
    X_test_processed.to_numpy(dtype="float32")
)

In [91]:
y_train_tensor = torch.tensor(
    y_train.to_numpy(dtype="float32")
).reshape(-1, 1)

In [92]:
y_val_tensor = torch.tensor(
    y_val.to_numpy(dtype="float32")
).reshape(-1, 1)

y_test_tensor = torch.tensor(
    y_test.to_numpy(dtype="float32")
).reshape(-1, 1)

In [93]:
print(X_train_tensor.shape)
print(y_train_tensor.shape)

print(X_train_tensor.dtype)
print(y_train_tensor.dtype)

torch.Size([1619, 17])
torch.Size([1619, 1])
torch.float32
torch.float32


In [94]:
torch.isnan(X_train_tensor).sum()

tensor(0)

In [95]:
X_train_tensor[:2]

tensor([[-0.8572, -0.2584,  2.0549, -0.0638, -0.6936, -0.5961, -0.3721, -0.1957,
         -0.4501, -0.5058,  0.0000,  0.0000,  0.0000,  0.0000,  1.0000,  0.0000,
          1.0000],
        [-1.0599, -0.1342, -0.6559,  0.2249,  0.1023,  0.5622,  0.0474,  0.2764,
         -0.0623,  1.8225,  0.0000,  0.0000,  0.0000,  0.0000,  1.0000,  1.0000,
          0.0000]])

In [96]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

In [97]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

In [98]:
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [99]:
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [100]:
X_batch, y_batch = next(iter(train_loader))

print(X_batch.shape)
print(y_batch.shape)

torch.Size([32, 17])
torch.Size([32, 1])


In [101]:
import torch.nn as nn

In [102]:
input_size = X_train_tensor.shape[1]

print(input_size)

17


In [103]:
model = nn.Sequential(
    nn.Linear(input_size, 32),
    nn.ReLU(),

    nn.Linear(32, 16),
    nn.ReLU(),

    nn.Linear(16, 1)
)

In [104]:
loss_fn = nn.MSELoss()

In [105]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [106]:
X_batch, y_batch = next(iter(train_loader))

In [107]:
num_epochs = 50

for epoch in range(num_epochs):

    # ----- training -----
    model.train()
    total_train_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        pred = model(X_batch)
        loss = loss_fn(pred, y_batch)

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item() * X_batch.size(0)

    average_train_loss = (
        total_train_loss / len(train_dataset)
    )

    # ----- validation -----
    model.eval()
    total_val_loss = 0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            pred = model(X_batch)
            loss = loss_fn(pred, y_batch)

            total_val_loss += (
                loss.item() * X_batch.size(0)
            )

    average_val_loss = (
        total_val_loss / len(val_dataset)
    )

    if (epoch + 1) % 10 == 0:
        print(
            "Epoch:", epoch + 1,
            "| Train:", average_train_loss,
            "| Val:", average_val_loss
        )

Epoch: 10 | Train: 1.3270981679218945 | Val: 1.3116289097585796
Epoch: 20 | Train: 1.2816839754912792 | Val: 1.3296537502312367
Epoch: 30 | Train: 1.2369926957306558 | Val: 1.3638932048538586
Epoch: 40 | Train: 1.181287699013746 | Val: 1.3914618009402429
Epoch: 50 | Train: 1.1230941975536428 | Val: 1.4647004998760458


In [108]:
small_model = nn.Sequential(
    nn.Linear(input_size, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

In [109]:
loss_fn = nn.MSELoss()

optimizer = torch.optim.Adam(
    small_model.parameters(),
    lr=0.001
)

In [110]:
import copy

best_val_mae = float("inf")
best_model_state = None
best_epoch = None

patience = 10
epochs_without_improvement = 0

num_epochs = 200

In [111]:
for epoch in range(num_epochs):

    # ----- training -----
    small_model.train()

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        pred = small_model(X_batch)
        loss = loss_fn(pred, y_batch)

        loss.backward()
        optimizer.step()

    # ----- validation -----
    small_model.eval()

    total_absolute_error = 0
    total_n = 0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            pred = small_model(X_batch)

            total_absolute_error += (
                torch.abs(pred - y_batch).sum().item()
            )

            total_n += y_batch.size(0)

    val_mae = total_absolute_error / total_n

    # ----- save best model -----
    if val_mae < best_val_mae:

        best_val_mae = val_mae
        best_epoch = epoch + 1

        best_model_state = copy.deepcopy(
            small_model.state_dict()
        )

        epochs_without_improvement = 0

    else:
        epochs_without_improvement += 1

    if (epoch + 1) % 10 == 0:
        print(
            "Epoch:", epoch + 1,
            "| Val MAE:", val_mae
        )

    if epochs_without_improvement >= patience:
        print("Early stopping at epoch", epoch + 1)
        break

Epoch: 10 | Val MAE: 0.8892474092083212
Early stopping at epoch 11


In [112]:
small_model.load_state_dict(best_model_state)

print("Best epoch:", best_epoch)
print("Best validation MAE:", best_val_mae)

Best epoch: 1
Best validation MAE: 0.8818077982207875


In [113]:
small_model.eval()

val_predictions = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        pred = small_model(X_batch)
        val_predictions.append(pred)

val_predictions = torch.cat(val_predictions).numpy().flatten()

In [114]:
mlp_mae = mean_absolute_error(
    y_val,
    val_predictions
)

mlp_r2 = r2_score(
    y_val,
    val_predictions
)

print("MLP MAE:", mlp_mae)
print("MLP R²:", mlp_r2)

MLP MAE: 0.8818077938844479
MLP R²: -0.00420762521510043


In [115]:
def train_mlp(seed, train_loader, val_loader):

    torch.manual_seed(seed)

    model = nn.Sequential(
        nn.Linear(input_size, 8),
        nn.ReLU(),
        nn.Linear(8, 1)
    )

    loss_fn = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    best_val_mae = float("inf")
    best_model_state = None
    best_epoch = None

    patience = 10
    epochs_without_improvement = 0

    for epoch in range(200):

        model.train()

        for X_batch, y_batch in train_loader:

            optimizer.zero_grad()

            pred = model(X_batch)
            loss = loss_fn(pred, y_batch)

            loss.backward()
            optimizer.step()

        model.eval()

        total_absolute_error = 0
        total_n = 0

        with torch.no_grad():

            for X_batch, y_batch in val_loader:

                pred = model(X_batch)

                total_absolute_error += (
                    torch.abs(pred - y_batch).sum().item()
                )

                total_n += y_batch.size(0)

        val_mae = total_absolute_error / total_n

        if val_mae < best_val_mae:

            best_val_mae = val_mae
            best_epoch = epoch + 1

            best_model_state = copy.deepcopy(
                model.state_dict()
            )

            epochs_without_improvement = 0

        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_model_state)

    return best_val_mae, best_epoch, model

In [116]:
seeds = [1, 2, 3, 4, 5]

mlp_results = []

for seed in seeds:

    mae, best_epoch, model = train_mlp(seed, train_loader, val_loader)

    mlp_results.append(mae)

    print(
        "Seed:", seed,
        "| Best Val MAE:", mae,
        "| Best Epoch:", best_epoch
    )

Seed: 1 | Best Val MAE: 0.8691053790810668 | Best Epoch: 2
Seed: 2 | Best Val MAE: 0.8771321520393277 | Best Epoch: 11
Seed: 3 | Best Val MAE: 0.8914729083025897 | Best Epoch: 5
Seed: 4 | Best Val MAE: 0.8840938756495347 | Best Epoch: 7
Seed: 5 | Best Val MAE: 0.8839034186469183 | Best Epoch: 7


In [117]:
y_train_abs = train_data["discordance"].copy()
y_val_abs = val_data["discordance"].copy()
y_test_abs = test_data["discordance"].copy()

In [118]:
y_train_abs_tensor = torch.tensor(
    y_train_abs.to_numpy(dtype="float32")
).reshape(-1, 1)

y_val_abs_tensor = torch.tensor(
    y_val_abs.to_numpy(dtype="float32")
).reshape(-1, 1)

In [119]:
train_abs_dataset = TensorDataset(
    X_train_tensor,
    y_train_abs_tensor
)

val_abs_dataset = TensorDataset(
    X_val_tensor,
    y_val_abs_tensor
)

train_abs_loader = DataLoader(
    train_abs_dataset,
    batch_size=32,
    shuffle=True
)

val_abs_loader = DataLoader(
    val_abs_dataset,
    batch_size=32,
    shuffle=False
)

In [120]:
dummy_abs_model = DummyRegressor(strategy="mean")

dummy_abs_model.fit(
    X_train_processed,
    y_train_abs
)

,"strategy strategy: {""mean"", ""median"", ""quantile"", ""constant""}, default=""mean""Strategy to use to generate predictions.* ""mean"": always predicts the mean of the training set* ""median"": always predicts the median of the training set* ""quantile"": always predicts a specified quantile of the training set, provided with the quantile parameter.* ""constant"": always predicts a constant value that is provided by the user.",'mean'
,"constant constant: int or float or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
,"quantile quantile: float in [0.0, 1.0], default=NoneThe quantile to predict using the ""quantile"" strategy. A quantile of0.5 corresponds to the median, while 0.0 to the minimum and 1.0 to themaximum.",None
Name,Type,Value
"constant_ constant_: ndarray of shape (1, n_outputs)Mean or median or quantile of the training targets or constant valuegiven by the user.","ndarray[float64](1, 1)",[[0.91]]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](17,)","['WBC','lymphocyte_pct','monocyte_pct',...,'smoking_status_never', 'sex_female','sex_male']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,17
n_outputs_ n_outputs_: intNumber of outputs.,int,1


In [121]:
y_val_abs_pred_dummy = dummy_abs_model.predict(
    X_val_processed
)

In [122]:
dummy_abs_mae = mean_absolute_error(
    y_val_abs,
    y_val_abs_pred_dummy
)

dummy_abs_r2 = r2_score(
    y_val_abs,
    y_val_abs_pred_dummy
)

print("Absolute Dummy MAE:", dummy_abs_mae)
print("Absolute Dummy R²:", dummy_abs_r2)

Absolute Dummy MAE: 0.5776217391103591
Absolute Dummy R²: -0.0015235278099312577


In [123]:
ridge_abs_model = Ridge(alpha=1.0)

In [124]:
ridge_abs_model.fit(
    X_train_processed,
    y_train_abs
)

,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' 

In [125]:
for alpha in alphas:
    model = Ridge(alpha=alpha)

    model.fit(
        X_train_processed,
        y_train_abs
    )

    pred = model.predict(
        X_val_processed
    )

    mae = mean_absolute_error(y_val_abs, pred)
    r2 = r2_score(y_val_abs, pred)

    print(
        "alpha =", alpha,
        "| MAE =", mae,
        "| R² =", r2
    )

alpha = 0.01 | MAE = 0.5814071437354824 | R² = -0.018058863044947193
alpha = 0.1 | MAE = 0.5813823579397365 | R² = -0.017919639138384236
alpha = 1 | MAE = 0.5812244074157487 | R² = -0.016684054873196796
alpha = 10 | MAE = 0.5807798507338134 | R² = -0.014074217579243653
alpha = 100 | MAE = 0.5805001373473411 | R² = -0.012716862369218651


In [126]:
y_train_abs_tensor = torch.tensor(
    y_train_abs.to_numpy(dtype="float32")
).reshape(-1, 1)

y_val_abs_tensor = torch.tensor(
    y_val_abs.to_numpy(dtype="float32")
).reshape(-1, 1)

In [127]:
train_abs_dataset = TensorDataset(
    X_train_tensor,
    y_train_abs_tensor
)

val_abs_dataset = TensorDataset(
    X_val_tensor,
    y_val_abs_tensor
)

In [128]:
train_abs_loader = DataLoader(
    train_abs_dataset,
    batch_size=32,
    shuffle=True
)

val_abs_loader = DataLoader(
    val_abs_dataset,
    batch_size=32,
    shuffle=False
)

In [129]:
abs_mlp_results = []

for seed in seeds:

    mae, best_epoch, model_abs = train_mlp(
        seed,
        train_abs_loader,
        val_abs_loader
    )

    abs_mlp_results.append(mae)

    print(
        "Seed:", seed,
        "| Best Val MAE:", mae,
        "| Best Epoch:", best_epoch
    )

Seed: 1 | Best Val MAE: 0.5731099164044415 | Best Epoch: 2
Seed: 2 | Best Val MAE: 0.5808214564382294 | Best Epoch: 4
Seed: 3 | Best Val MAE: 0.5841567074811017 | Best Epoch: 24
Seed: 4 | Best Val MAE: 0.5793669005970896 | Best Epoch: 10
Seed: 5 | Best Val MAE: 0.5854392251850646 | Best Epoch: 17


In [130]:
y_test_pred_dummy = dummy_model.predict(
    X_test_processed
)

dummy_test_mae = mean_absolute_error(
    y_test,
    y_test_pred_dummy
)

dummy_test_r2 = r2_score(
    y_test,
    y_test_pred_dummy
)

print("Dummy Test MAE:", dummy_test_mae)
print("Dummy Test R²:", dummy_test_r2)

Dummy Test MAE: 0.9518453690848349
Dummy Test R²: -0.004559905313263446


In [131]:
ridge_final = Ridge(alpha=100)

ridge_final.fit(
    X_train_processed,
    y_train
)

y_test_pred_ridge = ridge_final.predict(
    X_test_processed
)

ridge_test_mae = mean_absolute_error(
    y_test,
    y_test_pred_ridge
)

ridge_test_r2 = r2_score(
    y_test,
    y_test_pred_ridge
)

print("Ridge Test MAE:", ridge_test_mae)
print("Ridge Test R²:", ridge_test_r2)

Ridge Test MAE: 0.9528919630547701
Ridge Test R²: -0.004858697274108925


In [132]:
mlp_test_maes = []
mlp_test_r2s = []

for seed in seeds:

    val_mae, best_epoch, model = train_mlp(
        seed,
        train_loader,
        val_loader
    )

    model.eval()

    test_predictions = []

    with torch.no_grad():

        for X_batch, y_batch in test_loader:

            pred = model(X_batch)
            test_predictions.append(pred)

    test_predictions = (
        torch.cat(test_predictions)
        .numpy()
        .flatten()
    )

    test_mae = mean_absolute_error(
        y_test,
        test_predictions
    )

    test_r2 = r2_score(
        y_test,
        test_predictions
    )

    mlp_test_maes.append(test_mae)
    mlp_test_r2s.append(test_r2)

    print(
        "Seed:", seed,
        "| Test MAE:", test_mae,
        "| Test R²:", test_r2
    )

Seed: 1 | Test MAE: 0.9666204035177928 | Test R²: -0.018188130406879832
Seed: 2 | Test MAE: 0.9556108193003292 | Test R²: -0.0061500684896316216
Seed: 3 | Test MAE: 0.9515011070682027 | Test R²: -0.002116982187933214
Seed: 4 | Test MAE: 0.9613813209355351 | Test R²: -0.021299509411040862
Seed: 5 | Test MAE: 0.9471798069238545 | Test R²: -0.0024549892940199936


In [133]:
y_test_abs_pred_dummy = dummy_abs_model.predict(
    X_test_processed
)

dummy_abs_test_mae = mean_absolute_error(
    y_test_abs,
    y_test_abs_pred_dummy
)

dummy_abs_test_r2 = r2_score(
    y_test_abs,
    y_test_abs_pred_dummy
)

print("Dummy Test MAE:", dummy_abs_test_mae)
print("Dummy Test R²:", dummy_abs_test_r2)

Dummy Test MAE: 0.5921382911184722
Dummy Test R²: -0.0023915641039740088


In [134]:
ridge_abs_final = Ridge(alpha=100)

ridge_abs_final.fit(
    X_train_processed,
    y_train_abs
)

y_test_abs_pred_ridge = ridge_abs_final.predict(
    X_test_processed
)

ridge_abs_test_mae = mean_absolute_error(
    y_test_abs,
    y_test_abs_pred_ridge
)

ridge_abs_test_r2 = r2_score(
    y_test_abs,
    y_test_abs_pred_ridge
)

print("Ridge Test MAE:", ridge_abs_test_mae)
print("Ridge Test R²:", ridge_abs_test_r2)

Ridge Test MAE: 0.5950732741972518
Ridge Test R²: -0.014787747834930975


In [135]:
y_test_abs_tensor = torch.tensor(
    y_test_abs.to_numpy(dtype="float32")
).reshape(-1, 1)

test_abs_dataset = TensorDataset(
    X_test_tensor,
    y_test_abs_tensor
)

test_abs_loader = DataLoader(
    test_abs_dataset,
    batch_size=32,
    shuffle=False
)

In [136]:
mlp_abs_test_maes = []
mlp_abs_test_r2s = []

for seed in seeds:

    val_mae, best_epoch, model = train_mlp(
        seed,
        train_abs_loader,
        val_abs_loader
    )

    model.eval()

    test_abs_predictions = []

    with torch.no_grad():

        for X_batch, y_batch in test_abs_loader:

            pred = model(X_batch)
            test_abs_predictions.append(pred)

    test_abs_predictions = (
        torch.cat(test_abs_predictions)
        .numpy()
        .flatten()
    )

    test_abs_mae = mean_absolute_error(
        y_test_abs,
        test_abs_predictions
    )

    test_abs_r2 = r2_score(
        y_test_abs,
        test_abs_predictions
    )

    mlp_abs_test_maes.append(test_abs_mae)
    mlp_abs_test_r2s.append(test_abs_r2)

    print(
        "Seed:", seed,
        "| Test MAE:", test_abs_mae,
        "| Test R²:", test_abs_r2
    )

Seed: 1 | Test MAE: 0.6166391256836314 | Test R²: -0.16580455189345744
Seed: 2 | Test MAE: 0.607347661702467 | Test R²: -0.11937658197839296
Seed: 3 | Test MAE: 0.5902098490355079 | Test R²: -0.009069856172532642
Seed: 4 | Test MAE: 0.5871477381569461 | Test R²: -0.021360690166732255
Seed: 5 | Test MAE: 0.5939980788856921 | Test R²: -0.01906948227371008


In [137]:
import numpy as np

In [138]:
results = pd.DataFrame({
    "Outcome": [
        "Signed", "Signed", "Signed",
        "Absolute", "Absolute", "Absolute"
    ],
    "Model": [
        "Dummy", "Ridge", "MLP",
        "Dummy", "Ridge", "MLP"
    ],
    "Test_MAE": [
        dummy_test_mae,
        ridge_test_mae,
        np.mean(mlp_test_maes),

        dummy_abs_test_mae,
        ridge_abs_test_mae,
        np.mean(mlp_abs_test_maes)
    ],
    "Test_R2": [
        dummy_test_r2,
        ridge_test_r2,
        np.mean(mlp_test_r2s),

        dummy_abs_test_r2,
        ridge_abs_test_r2,
        np.mean(mlp_abs_test_r2s)
    ]
})

results

,Outcome,Model,Test_MAE,Test_R2
0,Signed,Dummy,0.951845,-0.004560
1,Signed,Ridge,0.952892,-0.004859
2,Signed,MLP,0.956459,-0.010042
3,Absolute,Dummy,0.592138,-0.002392
4,Absolute,Ridge,0.595073,-0.014788
5,Absolute,MLP,0.599068,-0.066936
